# Week 2 · Day 4 — Advanced CTEs: Chains, Scalars, and Snowflake-Safe Patterns

**Business First Prompt:**  
The logistics ops team needs layered reporting that can't be built in a single query — they want fleet-level benchmarks, driver comparisons against those benchmarks, and route-level efficiency scores all in one place. Your job today is to structure multi-CTE pipelines that are clean, readable, and would survive a code review in Snowflake or PostgreSQL.

**KPI Framework:**  
- Fleet average delivery time (scalar benchmark)
- Driver performance vs fleet average (deviation analysis)
- Route efficiency score (shipment volume + revenue per route)
- Warehouse throughput tier (HIGH / MID / LOW classification)

**What you'll build today:**
1. Multi-CTE chains — how CTEs see each other, column forwarding rules
2. Scalar CTEs + CROSS JOIN — single-value benchmarks used in downstream math
3. Window functions inside CTEs (the safe pattern for alias reuse)
4. Snowflake-safe guardrails — what SQLite lets you get away with that production won't
5. AI Audit drill — spot what would break in Snowflake

---

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'C:\Users\gianc\Documents\Logistics Operations Project\logistics_w2.db')
print('Connected to logistics_w2.db')

Connected to logistics_w2.db


In [3]:
for table in ['shipments', 'drivers', 'routes', 'warehouses', 'clients', 'vehicles', 'shipment_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- shipments ---
Columns: ['shipment_id', 'client_id', 'route_id', 'driver_id', 'vehicle_id', 'ship_date', 'delivery_date', 'status', 'freight_charge']


,shipment_id,client_id,route_id,driver_id,vehicle_id,ship_date,delivery_date,status,freight_charge
0,1001,4,4,3,2,2024-11-23,2024-11-24,Delivered,3375.74
1,1002,2,10,7,1,2024-12-12,None,Delayed,4030.98
2,1003,10,1,9,4,2024-01-16,2024-01-20,Delivered,1162.07



--- drivers ---
Columns: ['driver_id', 'name', 'province', 'hire_date', 'status']


,driver_id,name,province,hire_date,status
0,1,Jean Tremblay,QC,2019-03-15,Active
1,2,Mike Okafor,ON,2020-07-01,Active
2,3,Sara Singh,BC,2018-11-20,Active



--- routes ---
Columns: ['route_id', 'origin_wh_id', 'dest_city', 'dest_province', 'distance_km']


...

,client_id,company,city,province,segment
0,1,Maple Retail Co,Toronto,ON,Enterprise
1,2,Pacific Goods Ltd,Vancouver,BC,Enterprise
2,3,Prairie Supply Inc,Calgary,AB,SMB



--- vehicles ---
Columns: ['vehicle_id', 'plate', 'type', 'max_kg']


,vehicle_id,plate,type,max_kg
0,1,ON-A001,Van,500.0
1,2,ON-A002,Van,500.0
2,3,BC-B001,Truck,3000.0



--- shipment_items ---
Columns: ['item_id', 'shipment_id', 'description', 'category', 'weight_kg', 'declared_value']


,item_id,shipment_id,description,category,weight_kg,declared_value
0,1,1001,Electronics,Standard,787.6,321.17
1,2,1002,Office Supplies,Standard,614.3,3832.60
2,3,1002,Clothing,Standard,97.7,2512.06


---
## Concept 1 — Multi-CTE Chains: Column Visibility Rules

### What it is
A CTE chain is when CTE_B reads from CTE_A, and CTE_C reads from CTE_B. Each CTE only sees what the previous one explicitly selected — there is no reach-back to earlier layers.

### How it fits in clause order
```
WITH cte_a AS (...),
     cte_b AS (SELECT ... FROM cte_a),   -- can reference cte_a
     cte_c AS (SELECT ... FROM cte_b)    -- can reference cte_a OR cte_b
SELECT ... FROM cte_c
```
All CTEs in a WITH block are visible to every CTE defined after them — not just the immediate prior one.

### Common mistakes
- Selecting a column in CTE_A but forgetting to forward it through CTE_B — then trying to use it in CTE_C. It's gone.
- Adding columns to a downstream CTE that were never selected upstream.
- Confusing CTE chain (output flows forward) with parallel CTEs (independent, joined at the end).

### When NOT to use a chain
If CTE_B doesn't actually need any computed columns from CTE_A — it just needs raw table data — skip the chain and read from the base table directly.

### Worked example (real column names)
```sql
WITH driver_count AS (
    SELECT s.driver_id,
           d.name,
           d.province,
           COUNT(s.shipment_id) AS total_shipments
    FROM shipments s
    JOIN drivers d ON s.driver_id = d.driver_id
    GROUP BY s.driver_id, d.name, d.province
),
ranked AS (
    SELECT driver_id,
           name,
           province,
           total_shipments,
           DENSE_RANK() OVER (PARTITION BY province ORDER BY total_shipments DESC) AS province_rank
    FROM driver_count
)
SELECT driver_id, name, province, total_shipments, province_rank
FROM ranked
ORDER BY province, province_rank;
```
Note: `province` must be forwarded from `driver_count` into `ranked` — if it weren't selected in driver_count, the PARTITION BY in ranked would fail.

---

In [4]:
# q1 — Multi-CTE chain: driver shipment counts → ranked within province
# Build the two-CTE chain from the worked example above.
# Expected: driver_id, name, province, total_shipments, province_rank — ordered by province then rank.

q1 = """
WITH driver_count AS (  SELECT s.driver_id,
                                    d.name,
                                    d.province,
                                    COUNT(s.shipment_id) AS total_shipments
                            FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
                                    GROUP BY s.driver_id),
    
        rank AS ( SELECT  driver_id,
                    name,
                    province,
                    total_shipments,
                    DENSE_RANK() OVER (PARTITION BY province ORDER BY total_shipments DESC) AS region_rank
                    FROM driver_count
                    )
        SELECT driver_id,
                name,
                province,
                total_shipments,
                region_rank
        FROM rank
        ORDER BY province, region_rank

"""
df1 = pd.read_sql_query(q1, conn)
display(df1)

,driver_id,name,province,total_shipments,region_rank
0,4,Tom Beaumont,AB,22,1
1,3,Sara Singh,BC,27,1
2,7,Linda Park,BC,24,2
3,6,Carlos Rivera,MB,17,1
4,8,James Leblanc,NS,22,1
5,2,Mike Okafor,ON,21,1
6,5,Priya Mehta,ON,21,1
7,10,Ryan Kowalski,ON,13,2
8,9,Aisha Diallo,QC,23,1
9,1,Jean Tremblay,QC,10,2


In [5]:
# q2 — Three-CTE chain: shipment counts → ranked → top 1 per province
#
# CTE 1 (driver_count): driver_id, name, province, COUNT(shipment_id) AS total_shipments
#   JOIN drivers to shipments. GROUP BY driver_id, name, province.
#
# CTE 2 (ranked): driver_id, name, province, total_shipments,
#   ROW_NUMBER() OVER (PARTITION BY province ORDER BY total_shipments DESC) AS rn
#   FROM driver_count
#
# Final SELECT: driver_id, name, province, total_shipments WHERE rn = 1
# Order by province.

q2 = """
    WITH driver_count AS (  SELECT s.driver_id,
                                d.name,
                                d.province,
                                COUNT(s.shipment_id) AS shipment_count
                        FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id    
                                GROUP BY s.driver_id,d.name,d.province ),
    rank AS (   SELECT  driver_id,
                        name,
                        province,
                        shipment_count,
                        DENSE_RANK() OVER (PARTITION BY province ORDER BY shipment_count DESC) AS province_rank
                FROM driver_count
                )
    SELECT driver_id,
            name,
             province,
             shipment_count,
             province_rank
    FROM rank
    WHERE province_rank = 1
    ORDER BY province

"""
df2 = pd.read_sql_query(q2, conn)
display(df2)

,driver_id,name,province,shipment_count,province_rank
0,4,Tom Beaumont,AB,22,1
1,3,Sara Singh,BC,27,1
2,6,Carlos Rivera,MB,17,1
3,8,James Leblanc,NS,22,1
4,2,Mike Okafor,ON,21,1
5,5,Priya Mehta,ON,21,1
6,9,Aisha Diallo,QC,23,1


---
## Concept 2 — Scalar CTEs + CROSS JOIN

### What it is
A scalar CTE produces a single row (one or more benchmark values). CROSS JOIN attaches that row to every row in another table — no join key needed because there's only one row to attach.

### How it fits in clause order
```
WITH base AS (...),           -- multi-row aggregation
     benchmark AS (           -- scalar: one row, one or more columns
         SELECT AVG(metric) AS avg_val, MAX(metric) AS max_val
         FROM base
     )
SELECT b.*, bm.avg_val, bm.max_val,
       ROUND(100.0 * b.metric / bm.max_val, 1) AS pct_of_max
FROM base b
CROSS JOIN benchmark bm
ORDER BY b.metric DESC;
```

### Common mistakes
- Forgetting CROSS JOIN and getting a cartesian product warning (or just wrong row counts).
- Trying to compute the benchmark in the same layer as the base aggregation (nested aggregate — fails in Snowflake).
- Using a scalar subquery inline instead of a CTE — works but unreadable at scale.

### When NOT to use
If the benchmark varies by group (e.g., avg per province), it's not scalar — use a window function or a grouped join instead.

### Worked example (real column names)
```sql
WITH driver_revenue AS (
    SELECT driver_id, SUM(freight_charge) AS total_revenue
    FROM shipments
    GROUP BY driver_id
),
fleet_avg AS (
    SELECT AVG(total_revenue) AS avg_revenue
    FROM driver_revenue
)
SELECT dr.driver_id,
       dr.total_revenue,
       fa.avg_revenue,
       ROUND(dr.total_revenue - fa.avg_revenue, 2) AS deviation
FROM driver_revenue dr
CROSS JOIN fleet_avg fa
ORDER BY dr.total_revenue DESC;
```

---

In [6]:
# q3 — Scalar CTE: driver revenue vs fleet average
# Build the worked example above.
# Expected: driver_id, total_revenue, avg_revenue, deviation — ordered by total_revenue DESC.

q3 = """
WITH driver_revenue AS (    SELECT driver_id,
                                    SUM(freight_charge) AS total_revenue
                            FROM shipments
                            WHERE status = 'Delivered'
                            GROUP BY driver_id),
   fleet_avg AS (   SELECT AVG(total_revenue) AS avg_revenue
                    FROM driver_revenue
                )
    SELECT dr.driver_id,
            dr.total_revenue,
            fa.avg_revenue,
            ROUND(dr.total_revenue - fa.avg_revenue, 2) AS deviance
    FROM driver_revenue dr CROSS JOIN fleet_avg fa

"""
df3 = pd.read_sql_query(q3, conn)
display(df3)

,driver_id,total_revenue,avg_revenue,deviance
0,1,6682.19,24161.203,-17479.01
1,2,27143.77,24161.203,2982.57
2,3,37379.58,24161.203,13218.38
3,4,22524.42,24161.203,-1636.78
4,5,12719.59,24161.203,-11441.61
5,6,30453.34,24161.203,6292.14
6,7,33087.99,24161.203,8926.79
7,8,27825.29,24161.203,3664.09
8,9,34477.68,24161.203,10316.48
9,10,9318.18,24161.203,-14843.02


In [7]:
# q4 — Scalar CTE with two benchmarks: avg AND max shipments per driver


q4 = """

WITH driver_counts AS ( SELECT driver_id,
                                COUNT(shipment_id) AS shipment_count
                        FROM shipments
                        GROUP BY driver_id),
    benchmarks AS ( SELECT AVG(shipment_count) AS avg_shipment,
                            MAX(shipment_count) AS max_shipment
                    FROM driver_counts)
    SELECT dc.driver_id,
            dc.shipment_count,
            b.avg_shipment,
            b.max_shipment,
            ROUND(100.0 * dc.shipment_count / b.max_shipment, 1) AS pct_of_max
    FROM driver_counts dc CROSS JOIN benchmarks b
    ORDER BY dc.shipment_count DESC


"""
df4 = pd.read_sql_query(q4, conn)
display(df4)

,driver_id,shipment_count,avg_shipment,max_shipment,pct_of_max
0,3,27,20.0,27,100.0
1,7,24,20.0,27,88.9
2,9,23,20.0,27,85.2
3,4,22,20.0,27,81.5
4,8,22,20.0,27,81.5
5,2,21,20.0,27,77.8
6,5,21,20.0,27,77.8
7,6,17,20.0,27,63.0
8,10,13,20.0,27,48.1
9,1,10,20.0,27,37.0


---
## Concept 3 — Window Functions Inside CTEs

### What it is
Window functions (RANK, ROW_NUMBER, DENSE_RANK, LAG, LEAD) compute across rows without collapsing them. The safe pattern is to isolate them in their own CTE — never mix with GROUP BY in the same SELECT layer.

### How it fits in clause order
SQL execution order: FROM → WHERE → GROUP BY → HAVING → SELECT → WINDOW → ORDER BY.  
Window functions run after GROUP BY resolves. In the same SELECT layer, a window function can't reference an alias you just created — it hasn't resolved yet. Wrap in a CTE to use that alias downstream.

### Common mistakes
- Mixing `GROUP BY` and a window function in the same SELECT — SQLite may let this pass, Snowflake won't.
- Referencing a window function alias in a WHERE clause — use a CTE and filter in the outer SELECT.
- Using `PARTITION BY` with no column — always supply a column.
- Aliasing a window function result as `rank` — reserved word in PostgreSQL, alias as `rnk`.

### The fix: wrap in a CTE
```sql
-- ✅ Safe everywhere
WITH ranked AS (
    SELECT
        driver_id,
        province,
        total_revenue,
        RANK() OVER (PARTITION BY province ORDER BY total_revenue DESC) AS rnk
    FROM driver_stats
)
SELECT driver_id, province, total_revenue, rnk
FROM ranked
WHERE rnk <= 2;
```
The outer SELECT sees `rnk` as a regular column — it can filter, sort, compute, or join on it freely.

### PARTITION BY reminder
- `PARTITION BY province` → rank resets for each province ✅
- `PARTITION BY` with no column → **invalid syntax** — always supply a column ✅
- `RANK` is a reserved word in some dialects — alias it as `rnk` or `ranked` to be safe ✅

### The three window functions you need cold
| Function | What it does | Tie behaviour |
|---|---|---|
| `RANK()` | Rank with gaps (1,1,3) | Ties get same rank, next rank skips |
| `ROW_NUMBER()` | Unique sequential number | Ties broken arbitrarily |
| `DENSE_RANK()` | Rank without gaps (1,1,2) | Ties get same rank, no skip |

Use `RANK()` when you want to show ties. Use `ROW_NUMBER()` when you need exactly 1 row per partition (e.g., deduplication, top-N filtering).

### LAG/LEAD inside a CTE
```sql
WITH monthly AS (
    SELECT month, total_revenue,
           LAG(total_revenue) OVER (ORDER BY month) AS prev_revenue
    FROM monthly_totals
)
SELECT month, total_revenue, prev_revenue,
       ROUND(100.0 * (total_revenue - prev_revenue) / prev_revenue, 1) AS mom_pct
FROM monthly
WHERE prev_revenue IS NOT NULL;
```
The MoM calculation lives in the outer SELECT, where `prev_revenue` is just a column.

---

In [8]:
# q5 — top 2 drivers per province by freight revenue -- Window function in CTE, filter in outer SELECT


q5 = """
WITH driver_totals AS (  SELECT s.driver_id,
                                d.name,
                                d.province,
                                SUM(freight_charge) AS total_revenue
                        FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
                            WHERE s.status = 'Delivered'
                            GROUP BY s.driver_id, d.name, d.province),
    
    drivers_ranked AS ( SELECT  driver_id,
                                name,
                                province,
                                total_revenue,
                                RANK() OVER (PARTITION BY province ORDER BY total_revenue DESC) AS rnk
                        FROM driver_totals)
    SELECT driver_id,
         name,
        province,
         total_revenue,
         rnk
    FROM drivers_ranked
    WHERE rnk <= 2
    
                                
"""
df5 = pd.read_sql_query(q5, conn)
display(df5)

,driver_id,name,province,total_revenue,rnk
0,4,Tom Beaumont,AB,22524.42,1
1,3,Sara Singh,BC,37379.58,1
2,7,Linda Park,BC,33087.99,2
3,6,Carlos Rivera,MB,30453.34,1
4,8,James Leblanc,NS,27825.29,1
5,2,Mike Okafor,ON,27143.77,1
6,5,Priya Mehta,ON,12719.59,2
7,9,Aisha Diallo,QC,34477.68,1
8,1,Jean Tremblay,QC,6682.19,2


In [9]:
# q6 — ROW_NUMBER() for deduplication: one row per client (most recent shipment)

q6 = """
        WITH latest_shipment AS (   SELECT s.client_id,
                                            c.company,
                                            S.shipment_id,
                                            S.delivery_date,
                                            ROW_NUMBER() OVER (PARTITION BY s.client_id ORDER BY s.delivery_date) AS rnk                                        
                                    FROM shipments s JOIN clients c ON s.client_id = c.client_id
                                    WHERE delivery_date IS NOT NULL
                                    )
    SELECT  client_id,
            company,
             shipment_id,
            delivery_date
    FROM latest_shipment
    WHERE rnk = 1

"""
df6 = pd.read_sql_query(q6, conn)
display(df6)

,client_id,company,shipment_id,delivery_date
0,1,Maple Retail Co,1114,2024-02-11
1,2,Pacific Goods Ltd,1024,2024-05-06
2,3,Prairie Supply Inc,1150,2024-03-24
3,4,Atlantic Foods,1102,2024-01-21
4,5,Capital Services,1062,2024-01-20
5,6,Northern Freight,1143,2024-01-28
6,7,Rockies Imports,1115,2024-06-08
7,8,Maritime Wholesale,1038,2024-01-07
8,9,Crown Logistics,1175,2024-01-05
9,10,Lakeside Goods,1003,2024-01-20


In [10]:
# q7 — LAG inside CTE for month-over-month freight revenue

q7 = """
        WITH monthly_revenue AS (   SELECT strftime('%Y-%m', delivery_date) AS month,   
                                        SUM(freight_charge) AS total_revenue
                                    FROM shipments
                                    GROUP BY month
        ),
        
        lag AS ( SELECT  month,
                    total_revenue,
                    LAG(total_revenue) OVER (ORDER BY month ) AS prev
                   FROM monthly_revenue )
        
        SELECT month,
                total_revenue,
                prev,
                ROUND((total_revenue - prev)* 100.0 / prev, 1) AS mom_change
        FROM lag
    
"""
df7 = pd.read_sql_query(q7, conn)
display(df7)

,month,total_revenue,prev,mom_change
0,None,217190.12,NaN,NaN
1,2024-01,23419.09,217190.12,-89.2
2,2024-02,3376.72,23419.09,-85.6
3,2024-03,16515.01,3376.72,389.1
4,2024-04,12350.61,16515.01,-25.2
5,2024-05,24636.08,12350.61,99.5
6,2024-06,19201.40,24636.08,-22.1
7,2024-07,12385.60,19201.40,-35.5
8,2024-08,32559.99,12385.60,162.9
9,2024-09,14819.85,32559.99,-54.5


---
## Concept 4 — Snowflake-Safe Patterns: What SQLite Forgives

### Why this matters
SQLite is lenient. It lets you write patterns that would throw syntax or logic errors in Snowflake, PostgreSQL, and BigQuery — the databases you'll actually use in analyst roles. Writing Snowflake-safe SQL now means zero rework later.

### The four main traps

| Pattern | SQLite | Snowflake/PostgreSQL | Fix |
|---|---|---|---|
| Window function in same layer as GROUP BY | Often works | Fails — window must run after GROUP BY resolves | Two CTEs: GROUP BY first, then window |
| Referencing a window alias in same SELECT | Sometimes works | Always fails | Wrap in CTE, reference alias from outer SELECT |
| `AVG(SUM(col))` (nested aggregates) | Sometimes works | Always fails | CTE for inner agg, outer query for outer agg |
| `PARTITION BY` with no column | Sometimes silently ignores | Syntax error | Always supply a column |

### The safe mental model
Each SELECT layer should do **one thing**:
- Layer 1: joins + GROUP BY aggregations
- Layer 2: window functions on the aggregated output
- Layer 3: filters, final math, formatting

Never mix aggregations and window functions in the same SELECT. Never reference an alias you just created in the same SELECT.

### RANK vs reserved words
`RANK` is a reserved word in PostgreSQL and some Snowflake contexts. Always alias window function results as `rnk`, `ranked`, or similar — never leave the result unaliased or alias it as `rank`.

---

In [11]:
# q8 — Identify what's wrong
#
# The query below was written by a junior analyst. It runs in SQLite but would fail in Snowflake.
# Your task: DO NOT run it. Read it, identify ALL issues, and write the fixed version.
#
# Broken query:
# ---------------------------------------------------------------
# SELECT
#     d.province,
#     d.driver_id,
#     SUM(s.freight_charge) AS total_freight,
#     RANK() OVER (PARTITION BY ORDER BY total_freight DESC) AS rank
# FROM drivers d
# JOIN shipments s ON d.driver_id = s.driver_id
# GROUP BY d.province, d.driver_id;
# ---------------------------------------------------------------
#
# Issues to identify (write your answer as a comment before the fixed query):
#   Issue 1: ...
#   Issue 2: ...
#   Issue 3: ...
#
# Fix: rewrite as a two-CTE Snowflake-safe version.

q8 = """
-- Issue 1: There is an aggregate and a window function in SELECT.Having GROUP BY and a window function in the same layer breaks the query
-- Issue 2: Window function is referencing total_freight in SELECT
-- Issue 3: No column is referenced in the window function's PARTITION BY 

-- Fixed query:
WITH driver_freight AS ( SELECT s.driver_id,
                                d.province,
                                SUM(freight_charge) AS total_freight
                         FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
                         GROUP BY s.driver_id, d.province),
    
    ranking AS ( SELECT driver_id,
                        province,
                        total_freight,   
                        RANK() OVER (PARTITION BY province ORDER BY total_freight DESC ) AS rnk
                    FROM driver_freight)
    SELECT driver_id,
            province,
            total_freight,
            rnk   
    FROM ranking 
    

"""
df8 = pd.read_sql_query(q8, conn)
display(df8)

,driver_id,province,total_freight,rnk
0,4,AB,50640.03,1
1,3,BC,67894.50,1
2,7,BC,66159.88,2
3,6,MB,43517.73,1
4,8,NS,46532.18,1
5,2,ON,45217.99,1
6,5,ON,40334.07,2
7,10,ON,20923.24,3
8,9,QC,49249.43,1
9,1,QC,28333.10,2


In [12]:
# q9 — Warehouse throughput tiering with CASE WHEN + scalar benchmark
#
# Goal: classify each warehouse as HIGH / MID / LOW throughout vs the fleet average.

q9 = """
    WITH warehouse_volume AS (  SELECT  w.warehouse_id,
                                         w.name, 
                                        w.city,
                                        COUNT(s.shipment_id) AS shipment_count
                                
                                FROM shipments s JOIN routes r ON s.route_id = r.route_id 
                                JOIN warehouses w ON w.warehouse_id = r.origin_wh_id
                                GROUP BY w.warehouse_id, w.name, w.city ),

        avg_benchmark AS (  SELECT AVG(shipment_count) AS avg_shipment_count
                            FROM warehouse_volume)

     SELECT wv.warehouse_id, 
            wv.name, 
            wv.city,
            wv.shipment_count,
            ab.avg_shipment_count,
            CASE WHEN shipment_count >= avg_shipment_count * 1.2 THEN 'HIGH' WHEN shipment_count <= avg_shipment_count * 0.8 THEN 'LOW' ELSE 'MID'END AS tier
     FROM warehouse_volume wv CROSS JOIN avg_benchmark ab
      ORDER BY shipment_count  DESC                   
                                       
                                        
    
                
"""
df9 = pd.read_sql_query(q9, conn)
display(df9)

,warehouse_id,name,city,shipment_count,avg_shipment_count,tier
0,1,Toronto Central,Toronto,52,28.571429,HIGH
1,3,Calgary Depot,Calgary,42,28.571429,HIGH
2,2,Vancouver Hub,Vancouver,32,28.571429,MID
3,4,Montreal East,Montreal,31,28.571429,MID
4,5,Halifax Port,Halifax,19,28.571429,LOW
5,6,Winnipeg Cross,Winnipeg,14,28.571429,LOW
6,7,Ottawa Gov,Ottawa,10,28.571429,LOW


---
## Concept 5 — Parallel CTEs: When NOT to Chain

### What it is
Not every multi-CTE query is a chain. Sometimes CTEs are **independent aggregations** that you combine in a final JOIN — like assembling lego pieces rather than building a ladder.
### When to use parallel vs chain
- **Chain**: Output of CTE_A flows into CTE_B's logic (CTE_B needs the computed columns from A)
- **Parallel**: CTE_A and CTE_B are independent — different source tables, shared join key, combined at the end

### When NOT to use parallel CTEs
If both CTEs hit the same table with the same GROUP BY key, consolidate them into one CTE. Two CTEs on `shipments GROUP BY driver_id` is just redundant work — one CTE with both aggregations is cleaner and faster.

### Pattern (two different source tables)
```sql
WITH
driver_revenue AS (
    SELECT driver_id, SUM(freight_charge) AS total_revenue
    FROM shipments
    GROUP BY driver_id
),
driver_info AS (
    SELECT driver_id, name, province, hire_date
    FROM drivers
)
SELECT
    r.driver_id,
    i.name,
    i.province,
    i.hire_date,
    r.total_revenue
FROM driver_revenue r
JOIN driver_info i ON r.driver_id = i.driver_id
ORDER BY r.total_revenue DESC;
```

**Key point from Week 2 Day 3:** Same-grain aggregations (COUNT, SUM per same key) belong in **one** CTE — not separate CTEs joined back together. Use parallel CTEs when the two sides come from genuinely different sources.

---

In [13]:
# q10 — Parallel CTEs: driver revenue with info

q10 = """
        WITH driver_revenue AS ( SELECT driver_id,
                                        SUM(freight_charge) AS total_freight
                                FROM shipments
                                GROUP BY driver_id),
        driver_info AS ( SELECT driver_id,
                                name,
                                province,
                                hire_date,
                                status
                                 FROM drivers)
                    
        SELECT di.driver_id,
                di.name,
                di.province,
                di.hire_date,
                dr.total_freight
            

        FROM driver_revenue dr JOIN driver_info di ON dr.driver_id = di.driver_id
        ORDER BY dr.total_freight DESC 


"""
df12 = pd.read_sql_query(q10, conn)
display(df12)

,driver_id,name,province,hire_date,total_freight
0,3,Sara Singh,BC,2018-11-20,67894.50
1,7,Linda Park,BC,2019-01-08,66159.88
2,4,Tom Beaumont,AB,2021-02-10,50640.03
3,9,Aisha Diallo,QC,2021-11-15,49249.43
4,8,James Leblanc,NS,2023-03-01,46532.18
5,2,Mike Okafor,ON,2020-07-01,45217.99
6,6,Carlos Rivera,MB,2020-09-14,43517.73
7,5,Priya Mehta,ON,2022-05-30,40334.07
8,1,Jean Tremblay,QC,2019-03-15,28333.10
9,10,Ryan Kowalski,ON,2022-08-22,20923.24


---
## ★ Capstone — Driver Performance Scorecard

**Business ask:**  
The ops team wants a complete driver scorecard for Q-end review. For every driver, they need:
- Total shipments completed (delivered)
- Total freight revenue generated  
- Average delivery time (days)
- Revenue per shipment
- Rank within their province by revenue
- Deviation from fleet-average revenue per shipment
- A performance tier: **ELITE** (top 25% by revenue per shipment), **SOLID** (middle 50%), **REVIEW** (bottom 25%)

**Requirements:**
1. Use a minimum of 4 CTEs
2. Include at least one scalar CTE + CROSS JOIN
3. Window functions must be in their own CTE layer (not mixed with GROUP BY)
4. Must be Snowflake-safe (no alias reuse in same SELECT, no bare PARTITION BY)
5. Performance tier must use NTILE(4) window function — look it up in the hint below

**Hint — NTILE:**  
`NTILE(4) OVER (ORDER BY revenue_per_shipment DESC)` divides rows into 4 equal buckets (1 = top 25%, 4 = bottom 25%). You can use this to classify tiers with a CASE WHEN on the ntile value.


In [14]:
# ★ CAPSTONE — Driver Performance Scorecard
# Columns expected in final output:
#   driver_id, name, province, total_shipments, total_revenue,
#   avg_delivery_days, revenue_per_shipment, fleet_avg_rps, deviation,
#   province_rank, performance_tier

q_capstone = """

 WITH driver_totals AS (    SELECT s.driver_id,
                                    d.name,
                                    d.province,
                                    SUM(freight_charge) AS total_revenue,
                                    COUNT(shipment_id) AS shipment_count,
                                    ROUND(AVG(JULIANDAY(delivery_date) - JULIANDAY(ship_date)), 2) AS avg_delivery_days
                            FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
                            WHERE s.status = 'Delivered'
                            GROUP BY s.driver_id, d.province, d.name),
    province_rank AS (  SELECT driver_id,
                        name,
                        province,
                        total_revenue,
                        shipment_count,
                        avg_delivery_days,
                        RANK() OVER (PARTITION BY province ORDER BY total_revenue DESC) AS rnk
                        FROM driver_totals
                        ),
    fleet_avg AS (    SELECT  AVG(freight_charge) As fleet_avg_rps
                                FROM shipments
                                ),

    deviation AS (  SELECT  driver_id,
                        name,
                        province,
                        total_revenue,
                        shipment_count,
                        avg_delivery_days,
                        fleet_avg_rps,
                        rnk,
                             (total_revenue - fleet_avg_rps) AS rev_deviation
                              FROM province_rank dt CROSS JOIN fleet_avg),

      tiers AS ( SELECT driver_id,
                        name,
                        province,
                        total_revenue,
                        shipment_count,
                        avg_delivery_days,
                        fleet_avg_rps,
                        rnk,
                        rev_deviation,
                        CASE 
        WHEN NTILE(4) OVER (ORDER BY total_revenue DESC) = 1 THEN 'elite'
        WHEN NTILE(4) OVER (ORDER BY total_revenue DESC) = 2 THEN 'solid'
        ELSE 'review' END AS performance_tier
        FROM deviation)

    SELECT driver_id,
      name, 
      province,
        shipment_count,
          total_revenue,
          avg_delivery_days,  
          fleet_avg_rps, 
          rev_deviation,
          rnk,
          performance_tier
           
    FROM tiers 
    
"""
df_capstone = pd.read_sql_query(q_capstone, conn)
display(df_capstone)

,driver_id,name,province,shipment_count,total_revenue,avg_delivery_days,fleet_avg_rps,rev_deviation,rnk,performance_tier
0,3,Sara Singh,BC,15,37379.58,5.47,2294.01075,35085.56925,1,elite
1,9,Aisha Diallo,QC,15,34477.68,6.07,2294.01075,32183.66925,1,elite
2,7,Linda Park,BC,12,33087.99,4.83,2294.01075,30793.97925,2,elite
3,6,Carlos Rivera,MB,12,30453.34,5.50,2294.01075,28159.32925,1,solid
4,8,James Leblanc,NS,13,27825.29,6.08,2294.01075,25531.27925,1,solid
5,2,Mike Okafor,ON,13,27143.77,6.23,2294.01075,24849.75925,1,solid
6,4,Tom Beaumont,AB,9,22524.42,3.89,2294.01075,20230.40925,1,review
7,5,Priya Mehta,ON,9,12719.59,5.67,2294.01075,10425.57925,2,review
8,10,Ryan Kowalski,ON,8,9318.18,4.50,2294.01075,7024.16925,3,review
9,1,Jean Tremblay,QC,3,6682.19,4.33,2294.01075,4388.17925,2,review
